In [ ]:
%pip install torch==2.0.1 torchvision==0.15.2 --no-cache-dir --force-reinstall
%pip install bitsandbytes bert-score wandb huggingface-hub xformers hf_xet

import pandas as pd
import torch
from transformers.utils.quantization_config import BitsAndBytesConfig
from transformers.models.auto.processing_auto  import AutoProcessor
from transformers.models.auto.modeling_auto import AutoModelForVisualQuestionAnswering
from transformers.training_args_seq2seq import Seq2SeqTrainingArguments
from transformers.trainer_seq2seq import Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
from PIL import Image
import os
from sklearn.metrics import accuracy_score, f1_score
from bert_score import score as bert_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from tqdm import tqdm
import numpy as np
import re
# import wandb
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login, HfApi
import time

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 275.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 279.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 179.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 225.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 228.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 168.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 202.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 MB 247.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 MB 217.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.11/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-05-17 21:15:14.246167: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747516514.426560      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747516514.479918      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory 

In [7]:
# Paths (adjust <your-dataset-name> to your Kaggle dataset slug)
dataset_root = "/kaggle/input/"
vqa_csv_path = os.path.join(dataset_root, "vqa-dataset/vqa_dataset_final_new.csv")
image_metadata_path = os.path.join(dataset_root, "metadata/images.csv")
image_dir = os.path.join(dataset_root, "images/small")
output_path = "/kaggle/working/outputs/blip-vqa-base_finetuned.csv"
metrics_output_path = "/kaggle/working/outputs/blip-vqa-base_finetuned_metrics.csv"

os.makedirs("/kaggle/working/outputs", exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Set up WandB with Kaggle Secrets
# user_secrets = UserSecretsClient()
# wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
# os.environ["WANDB_API_KEY"] = wandb_api_key
# wandb.login()

# Check bf16 support
def is_bf16_supported():
    return torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 7  # T4/P100 support BF16

# Initialize WandB
# wandb.init(project="blip-vqa-finetuning", config={"model": "Salesforce/blip-vqa-base"})

wandb: Currently logged in as: pathaneni-anirudh (pathaneni-anirudh-international-institute-of-information) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


In [4]:
# Quantization config
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,  # 8-bit quantization
    bnb_8bit_compute_dtype=torch.bfloat16 if is_bf16_supported() else torch.float16,
    bnb_8bit_use_double_quant=True,  # Double quantization for memory savings
    bnb_8bit_quant_type="nf8"  # Normal 8-bit quantization
)

# Load model and processor
processor = AutoProcessor.from_pretrained("Salesforce/blip-vqa-base", useFast=True)
base_model = AutoModelForVisualQuestionAnswering.from_pretrained(
    "Salesforce/blip-vqa-base",
    torch_dtype=torch.bfloat16 if is_bf16_supported() else torch.float16,
    quantization_config=quantization_config if device.type == "cuda" else None,
)

# Try enabling xformers attention
try:
    base_model.config.use_xformers_attention = True
except AttributeError:
    print("xformers attention not supported, using default attention")

# Enable cache for inference
base_model.config.use_cache = True

# Set pad_token_id
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
base_model.config.pad_token_id = processor.tokenizer.pad_token_id

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

In [5]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value", "dense"],
    lora_dropout=0,
    bias="none",
    # task_type="SEQ_2_SEQ_LM"
)
model = get_peft_model(base_model, lora_config)

In [ ]:
try:
    vqa_data = pd.read_csv(vqa_csv_path)
    # Clean answers
    vqa_data["answer"] = vqa_data["answer"].fillna("unknown").astype(str).str.strip()
    vqa_data["answer"] = vqa_data["answer"].replace(r"^\s*$", "unknown", regex=True)
    vqa_data = vqa_data[vqa_data["answer"].apply(lambda x: len(x.strip().split()) > 0)]
    # Debug invalid answers
    invalid_answers = vqa_data[vqa_data["answer"].apply(lambda x: not x.strip() or len(x.strip().split()) == 0)]
    print(f"Invalid answers found: {len(invalid_answers)}")
    if len(invalid_answers) > 0:
        print(invalid_answers[["image_id", "question", "answer"]])
    image_metadata = pd.read_csv(image_metadata_path)[["image_id", "path"]]
except FileNotFoundError:
    raise FileNotFoundError("Dataset files not found. Ensure dataset is uploaded to /kaggle/input/vqa-dataset")
    
# Merge and filter data
vqa_data = vqa_data.merge(image_metadata, on="image_id", how="inner")
vqa_data["image_path"] = vqa_data["path"].apply(lambda x: os.path.join(image_dir, x))
vqa_data = vqa_data[vqa_data["image_path"].apply(os.path.exists)]

# Split data
train_data, val_data = train_test_split(vqa_data, test_size=0.1, random_state=3407)

Checking answers in vqa_data:
Invalid answers found: 0


In [ ]:
class VQADataset(Dataset):
    def __init__(self, data, processor, max_length=16):
        self.data = data
        self.processor = processor
        self.max_length = max_length
        self.load_times = []

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        start_time = time.time()
        row = self.data.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        question = row["question"]
        prompt = f"{question} Answer in one word."
        answer = str(row["answer"]).lower() if pd.notna(row["answer"]) else "unknown"

        inputs = self.processor(
            images=image,
            text=prompt,
            return_tensors="pt",
            padding="max_length",
            max_length=self.max_length,
            truncation=True
        )

        labels = self.processor.tokenizer(
            answer,
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        # Debug tokenized labels
        decoded_label = processor.tokenizer.decode(labels, skip_special_tokens=True)
        if not decoded_label.strip():
            print(f"Sample {idx}: Empty tokenized label: raw_answer='{answer}', decoded='{decoded_label}'")

        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        pixel_values = inputs["pixel_values"].squeeze(0)
        if isinstance(input_ids, list):
            input_ids = torch.tensor(input_ids, dtype=torch.long)
        if isinstance(attention_mask, list):
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        if isinstance(labels, list):
            labels = torch.tensor(labels, dtype=torch.long)

        load_time = time.time() - start_time
        self.load_times.append(load_time)

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

    def log_load_times(self):
        if self.load_times:
            avg_load_time = np.mean(self.load_times)
            # wandb.log({"avg_data_load_time": avg_load_time})
            print(f"Average data load time: {avg_load_time:.4f} seconds")

In [14]:
def custom_collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [10]:
# Create datasets
train_dataset = VQADataset(train_data, processor, max_length=16)
val_dataset = VQADataset(val_data, processor, max_length=16)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/outputs/results",
    run_name="blip-vqa-finetune",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # gradient_accumulation_steps=4,
    # warmup_steps=5, # 10
    num_train_epochs=3,
    learning_rate=1e-4,
    fp16=not is_bf16_supported(),
    bf16=is_bf16_supported(),
    optim="adamw_8bit",  # 8-bit AdamW for CUDA
    weight_decay=0.05,
    # lr_scheduler_type="cosine"
    seed=3407,
    report_to="wandb",
    logging_steps=100,
    logging_strategy="steps",
    save_strategy="epoch",
    load_best_model_at_end=True,
    remove_unused_columns=False,
    eval_strategy="epoch",
    metric_for_best_model="eval_accuracy",
    dataloader_num_workers=4,  # Optimized for Kaggle's 4-core CPUs
    predict_with_generate=True,
)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, np.ndarray):
        predictions = predictions
    else:
        predictions = predictions[0] if isinstance(predictions, tuple) else predictions
    decoded_preds = processor.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)
    
    preds = []
    cleaned_labels = []
    for i, (pred, label) in enumerate(zip(decoded_preds, decoded_labels)):
        pred_clean = pred.strip().lower()
        pred_words = pred_clean.split()
        pred_final = pred_words[0] if pred_words else "unknown"
        preds.append(pred_final)
        
        label_clean = label.strip().lower()
        label_words = label_clean.split()
        label_final = label_words[0] if label_words else "unknown"
        cleaned_labels.append(label_final)
        
        if not label_words:
            print(f"Eval sample {i}: Empty or invalid label: raw='{label}', cleaned='{label_clean}'")
    
    accuracy = accuracy_score(cleaned_labels, preds)
    f1 = f1_score(cleaned_labels, preds, average="macro")
    # wandb.log({"eval_accuracy": accuracy, "eval_f1": f1})
    return {"eval_accuracy": accuracy, "eval_f1": f1}

In [ ]:
def evaluate_model(model, data, processor, device):
    predictions = []
    ground_truths = []
    valid_indices = []
    generation_params = {
    "temperature": 0.3,
    "top_k": 40,
    "top_p": 0.8,
    "max_new_tokens": 5,
    "num_beams": 5,
    "no_repeat_ngram_size": 2,
    "use_cache": True, # KV Cache
    "do_sample": True
    }
    
    for idx, row in tqdm(data.iterrows(), total=len(data)):
        image = Image.open(row["image_path"]).convert("RGB")
        prompt = f"{row['question']} Answer in one word."
        ground_truth = str(row["answer"]).strip().lower().split()[0] if pd.notna(row["answer"]) else "unknown"

        inputs = processor(
            images=image,
            text=prompt,
            return_tensors="pt",
            padding="max_length",
            max_length=16,
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs["pixel_values"],
                **generation_params
            )
        predicted_answer = processor.decode(outputs[0], skip_special_tokens=True).strip().lower()
        predicted_answer = re.sub(r"[^\w\s]|'s|\?s", "", predicted_answer)
        if not predicted_answer or len(predicted_answer.split()) == 0 or len(predicted_answer) <= 1:
            predicted_answer = "unknown"
        else:
            predicted_answer = predicted_answer.split()[0]
            if len(predicted_answer) <= 1:
                predicted_answer = "unknown"

        predictions.append(predicted_answer)
        ground_truths.append(ground_truth)
        valid_indices.append(row.name)

    accuracy = accuracy_score(ground_truths, predictions)
    f1 = f1_score(ground_truths, predictions, average="macro")
    P, R, F1 = bert_score(predictions, ground_truths, lang="en")
    bertscore_f1 = F1.mean().item()

    metrics = {
        "accuracy": accuracy,
        "f1_score_macro": f1,
        "bertscore_f1": bertscore_f1,
        "num_evaluated": len(predictions)
    }
    # wandb.log(metrics)

    results_df = data.loc[valid_indices].copy()
    results_df["predicted_answer"] = predictions
    results_df["is_correct"] = [pred == gt for pred, gt in zip(predictions, ground_truths)]

    return results_df, metrics

In [ ]:
def main():
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        data_collator=custom_collate_fn
    )
    trainer.train()

    results_df, metrics = evaluate_model(model, val_data, processor, device)

    results_df[["image_id", "question", "answer", "predicted_answer", "is_correct"]].to_csv(output_path, index=False)
    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(metrics_output_path, index=False)

    model.save_pretrained("/kaggle/working/outputs/blip-vqa-base-lora-finetuned")
    processor.save_pretrained("/kaggle/working/outputs/blip-vqa-base-lora-finetuned")
    # wandb.save("/kaggle/working/outputs/blip-vqa-base-lora-finetuned/*")

    # wandb.finish()

if __name__ == "__main__":
    main()

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-mas

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,7.524600,7.522833,0.648241,0.250775
2,7.379500,7.386898,0.668342,0.274453
3,7.362800,7.382532,0.680219,0.293087


Sample 19486: Empty tokenized label: raw_answer='干邑色', decoded=''
Sample 15344: Empty tokenized label: raw_answer='മൾട്ടി', decoded=''
Sample 1190: Empty tokenized label: raw_answer='വെള്ള', decoded=''
Eval sample 1190: Empty or invalid label: raw='', cleaned=''


/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.11/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Sample 15344: Empty tokenized label: raw_answer='മൾട്ടി', decoded=''
Sample 19486: Empty tokenized label: raw_answer='干邑色', decoded=''
Sample 1190: Empty tokenized label: raw_answer='വെള്ള', decoded=''
Eval sample 1190: Empty or invalid label: raw='', cleaned=''


...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 8 files into the W&B run directory, call wandb.save again to sync new files.


accuracy,▁
bertscore_f1,▁
eval/accuracy,▁▅█
eval/f1,▁▅█
eval/loss,█▁▁
eval/runtime,█▁▁
eval/samples_per_second,▁██
eval/steps_per_second,▁██
eval_accuracy,▁▅█
eval_f1,▁▅█
f1_score_macro,▁


In [ ]:
# hf_token = ""
# login(hf_token)
# repo_id = "5unnySunny/blip-vqa-base"
# model.push_to_hub(repo_id, commit_message="Upload LoRA fine-tuned BLIP VQA model")
# processor.push_to_hub(repo_id, commit_message="Upload processor")
# print(f"Model and processor uploaded to https://huggingface.co/{repo_id}")
